# Contextual Retrieval 실습: 해운 관련 용어 RAG

### 1. 환경 설정 및 라이브러리 로드

다국어 환경과 BM25, Reranker를 모두 포함하는 설정

In [72]:
import os
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

load_dotenv()

# 모델 초기화
context_llm = ChatOpenAI(model="gpt-4.1-nano", temperature=0) # 컨텍스트 생성용으로, 가벼운 모델 사용
answer_llm = ChatOpenAI(model="gpt-4.1-mini", temperature=0)      # 최종 답변 생성용
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# 경로 설정
PDF_FILE_PATH = "data/BBC-Shipping-Chartering-Guide.pdf"
DB_PATH = "./chroma_db_shipping_chartering"

### 2. 문서 로드 및 청킹

In [6]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 1. 문서 로드
print(f"📄 PDF 파일({PDF_FILE_PATH})을 로드 중입니다...")
loader = PyPDFLoader(PDF_FILE_PATH)
docs = loader.load()

# 전체 문서 텍스트 결합 (컨텍스트 생성 시 참조용)
whole_document_text = "\n\n".join([doc.page_content for doc in docs])

# 2. 문서 청킹 (해운 실무 문서 최적화)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,        # 해운 실무 문장의 복잡성 고려
    chunk_overlap=150,     # 문맥 연결을 위해 겹치는 부분도 조금 늘림
    separators=["\n\n", "\n", ". ", " ", ""] # 표나 리스트 보존을 위해 세밀하게 설정
)
chunks = text_splitter.split_documents(docs)
print(f"생성된 청크 수: {len(chunks)}")

📄 PDF 파일(data/BBC-Shipping-Chartering-Guide.pdf)을 로드 중입니다...
생성된 청크 수: 186


### 3. Contextual Retrieval (맥락 생성)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.documents import Document

# 맥락 생성용 프롬프트
context_prompt = ChatPromptTemplate.from_messages([
    ("system", """당신은 해운 전문가입니다. 
제공된 청크의 맥락을 다음 규칙에 따라 한국어로 요약하세요:
- 문서의 어느 섹션(예: 목차, 용어집, 용선 계약 가이드 등)인지 명시할 것.
- 해당 청크가 다루는 구체적인 해운 주제(예: Agency Fee, 항만 비용 계산 등)를 포함할 것.
- "1. 이 청크는" 같은 서술어는 생략하고 핵심 키워드 중심으로 100자 이내로 작성할 것."""),
    ("user", "전체 문서 내용:\n{whole_document}\n\n분석할 청크:\n{chunk_content}")
])

context_chain = context_prompt | context_llm | StrOutputParser()

맥락이 잘 생성되는지 테스트

In [ ]:

# 테스트용 (30개만)
# 각 청크에 맥락 붙이기
print("맥락 생성 시작...")
inputs = [{"whole_document": whole_document_text, "chunk_content": c.page_content} for c in chunks]
contexts = context_chain.batch(inputs[:30]) # 우선 상위 30개만 테스트

contextual_chunks = []
for i, (chunk, context) in enumerate(zip(chunks[:30], contexts)):
    # 핵심: 본문에 맥락을 합칩니다
    new_content = f"[이 청크의 맥락: {context}]\n\n{chunk.page_content}"
    
    # 메타데이터 유지 (페이지 번호 등)
    new_doc = Document(
        page_content=new_content,
        metadata=chunk.metadata
    )
    contextual_chunks.append(new_doc)

print("맥락 결합 완료!")

맥락 생성 시작...
맥락 결합 완료!


In [14]:
# 20번부터 30번까지 확인 (리스트 인덱스 기준 19~29)
start_idx = 19
end_idx = 30

print(f"🔍 [20번~30번 청크] 상세 검토 (총 {end_idx - start_idx}개)\n")
print("=" * 80)

# 가공된 청크 리스트에서 해당 구간만 슬라이싱
for i, doc in enumerate(contextual_chunks[start_idx:end_idx]):
    current_num = start_idx + i + 1
    page_num = doc.metadata.get('page', 'N/A')
    
    # [Context: ...] 와 본문 분리
    content_parts = doc.page_content.split('\n\n')
    context_part = content_parts[0]
    body_part = content_parts[1] if len(content_parts) > 1 else ""
    
    print(f"[{current_num}번 청크] 📄 Page: {page_num}")
    print(f"💡 생성된 맥락: {context_part}")
    print(f"📝 본문 미리보기: {body_part[:200].replace('\n', ' ')}...") # 본문을 좀 더 길게(200자) 봅니다
    print("-" * 80)

print(f"\n✅ {start_idx+1}번부터 {end_idx}번까지 검토가 완료되었습니다.")

🔍 [20번~30번 청크] 상세 검토 (총 11개)

[20번 청크] 📄 Page: 10
💡 생성된 맥락: [이 청크의 맥락: 용어집 - 해운 용어: Bale, Ballast Bonus, Bareboat Charter 등 선박 용량, 보상, 선박 임대 관련 해운 용어 설명.]
📝 본문 미리보기: percentage of freight Bale - A measurement of the vessel’s carrying capacity, usually for breakbulk cargo, which  takes into consideration the inability to load between the vessel’s stanchions or fram...
--------------------------------------------------------------------------------
[21번 청크] 📄 Page: 10
💡 생성된 맥락: [이 청크의 맥락: 용선계약 가이드 - 본 청크는 Bareboat Charter(선박 임대) 정의와 Baltic Code(중개인 윤리규범) 설명 포함.]
📝 본문 미리보기: commencement of the charter party. Bareboat Charter – Arrangement for the charter of a ship in which the shipowner only provides  the ship; while the charterer provides the crew, stores, bunkers, and ...
--------------------------------------------------------------------------------
[22번 청크] 📄 Page: 11
💡 생성된 맥락: [이 청크의 맥락: 용어집; Baltic Exchange와 Balitime 관련 해운 시장 정보 및 표준 계약 소개]
📝 본문 미리보기: 12 Version 2.0 | July 2025 Bal

실제 전체 청크에 대한 맥락 생성

In [15]:
# 2. 전체 청크에 대한 입력값 준비
print(f"🚀 총 {len(chunks)}개 청크에 대한 맥락 생성을 시작합니다...")
inputs = [{"whole_document": whole_document_text, "chunk_content": c.page_content} for c in chunks]

# 3. 전체 데이터 배치 처리 (범위 제한 해제)
# max_concurrency는 API 속도 제한(Rate Limit)을 고려하여 5~10 사이를 추천합니다.
contexts = context_chain.batch(inputs, config={"max_concurrency": 8})

# 4. 맥락 결합 및 새로운 Document 리스트 생성
contextual_chunks = []
for i, (chunk, context) in enumerate(zip(chunks, contexts)):
    # 검색 성능을 위해 본문 맨 앞에 맥락 삽입
    new_content = f"[Context: {context}]\n\n{chunk.page_content}"
    
    new_doc = Document(
        page_content=new_content,
        metadata=chunk.metadata # 원본의 page, source 정보 유지
    )
    contextual_chunks.append(new_doc)

print(f"✅ {len(contextual_chunks)}개 청크 맥락 결합 완료!")

🚀 총 186개 청크에 대한 맥락 생성을 시작합니다...
✅ 186개 청크 맥락 결합 완료!


### 4. 디비에 저장

In [18]:
import os
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

# 1. 맥락 생성 (이미 생성된 contexts 사용)
print(f"🚀 총 {len(chunks)}개 청크 가공 및 Chroma DB 저장 시작...")

contextual_chunks = []
for i, (chunk, context) in enumerate(zip(chunks, contexts)):
    # 검색 성능 극대화를 위해 본문 앞에 맥락 삽입
    new_content = f"[Context: {context}]\n\n{chunk.page_content}"
    
    new_doc = Document(
        page_content=new_content,
        metadata=chunk.metadata # 원본 페이지 번호 등 유지
    )
    contextual_chunks.append(new_doc)

print(f"✅ {len(contextual_chunks)}개 청크 맥락 결합 완료!")

# 2. Chroma DB 저장
print(f"📦 '{DB_PATH}'에 데이터를 저장 중입니다. 잠시만 기다려주세요...")

vectorstore = Chroma.from_documents(
    documents=contextual_chunks, 
    embedding=embeddings,
    persist_directory=DB_PATH,
    collection_name="bbc_guide_v2"
)

print(f"✅ 완료! 이제 '{DB_PATH}' 폴더에 모든 데이터가 저장되었습니다.")

🚀 총 186개 청크 가공 및 Chroma DB 저장 시작...
✅ 186개 청크 맥락 결합 완료!
📦 './chroma_db_shipping_chartering'에 데이터를 저장 중입니다. 잠시만 기다려주세요...
✅ 완료! 이제 './chroma_db_shipping_chartering' 폴더에 모든 데이터가 저장되었습니다.


디비에 잘 적용되었는지 확인

In [ ]:
# 1. DB에 저장된 전체 데이터(청크) 개수 확인
collection = vectorstore._collection
total_count = collection.count()
print(f"📊 DB에 저장된 총 청크 수: {total_count}개")

# 2. 중간 인덱스 설정 (예: 전체의 절반 지점)
mid_idx = total_count // 2

print(f"\n🔍 [인덱스 {mid_idx}번] 데이터 확인 시작...")
print("=" * 80)

mid_data = collection.get(limit=1, offset=mid_idx)

if mid_data['documents']:
    content = mid_data['documents'][0]
    metadata = mid_data['metadatas'][0] if mid_data['metadatas'] else {}
    
    # [Context: ...] 부분이 잘 들어갔는지 확인
    print(f"📄 원본 페이지: {metadata.get('page', 'N/A')}")
    print(f"💡 저장된 내용 (앞부분 300자):\n{content[:300]}...")
else:
    print("❌ 해당 인덱스에 데이터가 없습니다.")

print("=" * 80)

📊 DB에 저장된 총 청크 수: 186개

🔍 [인덱스 93번] 데이터 확인 시작...
📄 원본 페이지: 33
💡 저장된 내용 (앞부분 300자):
[Context: 용어집 - 선적 용어, 적재량 조정, 선적량 분쟁 표시]

carry over or under a certain quantity of cargo specified in the voyage charter.; this is useful 
if the owner or carrier is uncertain what the ship’s cargo capacity will be upon loading.  
More in dispute if on board to be delivered - Notation made on a bi...


### 5. 하이브리드 검색(Hybrid Search) + 리랭킹(Reranking)

#### 파이프라인 구성 요약

* **Base**: **Dense Retrieval** (순수 Embedding) + **Context Info** (메타데이터로 청크 위치/주제 식별)
* **Hybrid**: **Ensemble Retrieval** (Embedding + BM25) + **Context Info**
* **Rerank**: **Final Re-ranking** (Hybrid 결과물을 Cross-Encoder로 재정렬) + **Context Info**

In [ ]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder

# 1. BM25 검색기 생성 (키워드 중심)
# contextual_chunks 리스트를 기반으로 만듭니다.
bm25_retriever = BM25Retriever.from_documents(contextual_chunks)
bm25_retriever.k = 5  # 상위 5개 추출

# 2. 벡터 검색기 생성 (의미 중심)
chroma_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# 3. 앙상블 검색기 (Hybrid: BM25 + Vector)
# 해운 용어 특성상 키워드(BM25) 비중 강화
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, chroma_retriever],
    weights=[0.7, 0.3]
)

# 4. 리랭커(Reranker) 설정 (BGE-Reranker 모델 사용)
# 이 모델은 검색된 결과들을 다시 읽어보고 질문과 가장 관련 있는 순서로 재정렬합니다.
model_name = "BAAI/bge-reranker-v2-m3" # 다국어 지원 모델
re_ranker_model = HuggingFaceCrossEncoder(model_name=model_name)
compressor = CrossEncoderReranker(model=re_ranker_model, top_n=3)

# 5. 최종 파이프라인 완성: Contextual Hybrid + Reranker
hybrid_rerank_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, 
    base_retriever=ensemble_retriever
)

print("🚀 하이브리드 + 리랭커 파이프라인 구성 완료!")

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5427.04it/s]


🚀 하이브리드 + 리랭커 파이프라인 구성 완료!


성능평가용 질문 목록

In [55]:
# BBC Shipping & Chartering Guide 성능 평가용 골든 셋 (총 20개)
eval_set = [
    # 1. 앞부분 (p.8 ~ p.12)
    {"q": "AAAA 조건에서 선박이 바닥에 닿으면 안 되는 이유는?", "expected": "not touch bottom"},
    {"q": "Agency Fee는 보통 누가 누구에게 지급하나요?", "expected": "paid to a port agent by the shipowner"},
    {"q": "Bareboat Charter와 일반 용선의 가장 큰 차이점은?", "expected": "charterer provides the crew"},
    {"q": "BBB는 언제 지급되는 비용인가요?", "expected": "payment of freight before discharging"},
    {"q": "Ballast Bonus가 지급되는 일반적인 상황은?", "expected": "delivery of a ship at a place other than where she was redelivered"},
    
    # 2. 중간 부분 (p.14 ~ p.32)
    {"q": "Cancellation Clause에 따라 용선자가 계약을 취소할 수 있는 조건은?", "expected": "vessel is not presented by the specified dates"},
    {"q": "Deadfreight는 어떤 경우에 발생하는 손해배상금인가요?", "expected": "failing to load the amount of cargo stipulated"},
    {"q": "Demurrage와 Detention의 결정적인 차이점은?", "expected": "determined by a court, not liquidated damages"},
    {"q": "FAC 조건이 의미하는 바는 무엇인가요?", "expected": "as fast as the ship can load/discharge"},
    {"q": "Freight의 정의는 무엇인가요?", "expected": "remuneration payable to the carrier"},
    {"q": "Liner Terms 조건에서 적재 및 양하 비용 책임은 누구에게 있나요?", "expected": "carrier is responsible for the costs of loading and discharging"},
    
    # 3. 뒷부분 (p.30 ~ p.46)
    {"q": "Ice Clause에서 선장이 가질 수 있는 권한은?", "expected": "right to divert the ship"},
    {"q": "Laycan 기간 중 선박이 캔슬링 데이트보다 늦게 오면 어떻게 되나요?", "expected": "charterer has the option of canceling"},
    {"q": "Off Hire 상황이 발생하는 대표적인 예시는?", "expected": "breakdown of the ship or her equipment"},
    {"q": "Redelivery 시 선박의 상태에 대한 규정은?", "expected": "in the same good order and condition as when delivered"},
    {"q": "Safe Berth의 책임은 누구에게 있나요?", "expected": "responsibility on the cargo interests to nominate"},
    {"q": "Time Charter 계약에서 용선자가 부담하는 연료 및 항구 비용은?", "expected": "pay for all fuel, port charges"},
    {"q": "WWD 조건에서 레이타임 계산 방식은?", "expected": "laytime will count unless bad weather halts operations"},
    
    # 4. 부록 및 인코텀즈 (p.80 ~ p.81)
    {"q": "Incoterms 2020 중 FAS 조건에서 위험 이전 시점은?", "expected": "goods are alongside the vessel"},
    {"q": "CFR 조건에서 매도인이 의무적으로 제공하지 않아도 되는 것은?", "expected": "no obligation to provide insurance"}
]

print(f"✅ 총 {len(eval_set)}개의 평가 데이터셋이 준비되었습니다.")

✅ 총 20개의 평가 데이터셋이 준비되었습니다.


성능평가 지표 계산 함수

In [56]:
import tiktoken

def count_tokens(text):
    try:
        enc = tiktoken.encoding_for_model("gpt-4.1-mini")
    except KeyError:
        # 아직 라이브러리에 4.1 이름이 직접 등록 안 되어 있을 경우를 대비한 폴백
        enc = tiktoken.get_encoding("o200k_base") 
        
    return len(enc.encode(text))


def evaluate_performance_detailed(retriever, test_set, k=5):
    hits = 0
    mrr_sum = 0
    total_tokens = 0
    
    for item in test_set:
        query = item['q']
        expected_raw = item['expected'].lower() # 원본 소문자 유지
        
        # 검색 수행
        docs = retriever.invoke(query)
        
        # 1. 상위 k개 문서의 토큰 합계 계산
        current_query_tokens = sum(count_tokens(doc.page_content) for doc in docs[:k])
        total_tokens += current_query_tokens
        
        # 2. Hit Rate & MRR 계산
        found = False
        # 비교를 위해 본문에서 공백 제거한 버전도 준비 (선택사항)
        for rank, doc in enumerate(docs[:k], 1):
            content = doc.page_content.lower()
            
            # [수정된 매칭 로직]: 원본 expected의 단어들이 본문에 있는지 체크
            # 4글자 이상의 핵심 키워드가 하나라도 있으면 Hit!
            keywords = [w for w in expected_raw.split() if len(w) > 3]
            if any(word in content for word in keywords):
                hits += 1
                mrr_sum += (1 / rank)
                found = True
                break
                
    num_tests = len(test_set)
    return {
        "HitRate": hits / num_tests,
        "MRR": mrr_sum / num_tests,
        "AvgTokens": total_tokens / num_tests
    }


단계별 성능 비교

In [83]:
def run_final_evaluation(test_set):
    # 결과를 저장할 딕셔너리
    results = {}
    retrievers = {
        'Base Vector': chroma_retriever,
        'Hybrid': ensemble_retriever,
        'Hybrid+Rerank': hybrid_rerank_retriever
    }

    print("🚀 GPT-4.1 기반 RAG 파이프라인 성능 평가 시작...")

    # 1. 지표 계산
    for name, retriever in retrievers.items():
        results[name] = evaluate_performance_detailed(retriever, test_set, k=5)

    # --- [최종 성능 평가 결과 보고서] ---
    print("\n📊 [최종 성능 평가 결과 보고서]")
    print("-" * 75)
    header = f"{'Mode':<20} | {'Hit Rate':<12} | {'MRR':<10} | {'Avg Tokens':<12}"
    print(header)
    print("-" * 75)

    for mode, m in results.items():
        print(f"🔹 {mode:<17} | {m['HitRate']:>10.2%} | {m['MRR']:>8.4f} | {m['AvgTokens']:>8.1f} tk")

    print("-" * 75)
    print(f"💡 Tip: Reranker 사용 시 최종 전달 청크 수(top_n)에 따라 Avg Tokens가 최적화됩니다.")

   # 3. 리트리버별 1순위 청크 전격 비교
    print(f"\n🧐 [리트리버별 Top-1 근거 비교 분석]")
    print("=" * 100)
    
    for i, item in enumerate(test_set, 1):
        query = item['q']
        expected = item['expected']
        expected_words = [w.lower() for w in expected.split() if len(w) > 3]

        print(f"[{i}] 질문: {query}")
        print(f"🎯 기대 키워드: {expected}")
        print("-" * 100)
        
        # 각 리트리버별로 결과를 뽑아서 비교
        for name, retriever in retrievers.items():
            retrieved_docs = retriever.invoke(query)
            top_chunk = retrieved_docs[0].page_content if retrieved_docs else "검색 결과 없음"
            
            # 매칭 판정
            is_hit = "✅ 일치" if any(w in top_chunk.lower() for w in expected_words) else "❌ 불일치"
            
            print(f"🔍 [{name:<13}] {is_hit}")
            print(f"   > {top_chunk[:200].replace('\\n', ' ')}...") 
        
        print("=" * 100)
    
    return results # 나중에 데이터 활용을 위해 결과 리턴

In [84]:
run_final_evaluation(eval_set)

🚀 GPT-4.1 기반 RAG 파이프라인 성능 평가 시작...

📊 [최종 성능 평가 결과 보고서]
---------------------------------------------------------------------------
Mode                 | Hit Rate     | MRR        | Avg Tokens  
---------------------------------------------------------------------------
🔹 Base Vector       |    100.00% |   0.9750 |    915.5 tk
🔹 Hybrid            |    100.00% |   0.8750 |    886.4 tk
🔹 Hybrid+Rerank     |    100.00% |   1.0000 |    592.5 tk
---------------------------------------------------------------------------
💡 Tip: Reranker 사용 시 최종 전달 청크 수(top_n)에 따라 Avg Tokens가 최적화됩니다.

🧐 [리트리버별 Top-1 근거 비교 분석]
[1] 질문: AAAA 조건에서 선박이 바닥에 닿으면 안 되는 이유는?
🎯 기대 키워드: not touch bottom
----------------------------------------------------------------------------------------------------
🔍 [Base Vector  ] ❌ 불일치
   > [Context: 용어집 섹션, 해운 분쟁 해결 규칙 및 도착 선박 정의 설명. 아비트라션 규칙과 도착 선박 조건에 관한 해운 계약 용어 설명.]

Arbitration Rules or Terms - The rules under which arbitration will be held. Arbitrations are 
frequently sub

{'Base Vector': {'HitRate': 1.0, 'MRR': 0.975, 'AvgTokens': 915.45},
 'Hybrid': {'HitRate': 1.0, 'MRR': 0.875, 'AvgTokens': 886.35},
 'Hybrid+Rerank': {'HitRate': 1.0, 'MRR': 1.0, 'AvgTokens': 592.55}}

난이도가 높은 테스트 질문 추가

In [85]:
# 1. 난이도 상 질문 리스트 (기준점)
extra_stress_set = [
    {"q": "Laycan 기간 중 Cancellation Clause를 행사하기 위한 선행 조건은?", "expected": "canceling date is not respected"},
    {"q": "Liner Terms와 Free Out 조건의 discharging 비용 부담 차이는?", "expected": "carrier is responsible for the costs in Liner Terms"},
    {"q": "Time Charter 도중 Off-Hire 상황 시 용선료와 연료비 처리는?", "expected": "suspension of hire and fuel consumed during that period"},
    {"q": "Safe Berth 접근 중 선박 바닥이 닿는 사고 발생 시 누구의 책임인가?", "expected": "responsibility on the cargo interests to nominate"},
    {"q": "Deadfreight와 Demurrage가 발생하는 근본적인 원인의 차이는?", "expected": "failing to load the amount vs exceeding the allowed time"}
]

# 2. 기준이 되는 extra_stress_set에 기존 eval_set(20개)를 뒤로 병합
# 이제 extra_stress_set가 총 25개의 질문을 가지게 됩니다.
extra_stress_set.extend(eval_set)

print(f"🔥 병합 완료! 이제 extra_stress_set가 메인 평가셋이 되었습니다.")
print(f"📊 구성 현황: 심화(앞순서 5개) + 기본(뒷순서 20개) = 총 {len(extra_stress_set)}개")

run_final_evaluation(extra_stress_set)

🔥 병합 완료! 이제 extra_stress_set가 메인 평가셋이 되었습니다.
📊 구성 현황: 심화(앞순서 5개) + 기본(뒷순서 20개) = 총 25개
🚀 GPT-4.1 기반 RAG 파이프라인 성능 평가 시작...

📊 [최종 성능 평가 결과 보고서]
---------------------------------------------------------------------------
Mode                 | Hit Rate     | MRR        | Avg Tokens  
---------------------------------------------------------------------------
🔹 Base Vector       |    100.00% |   0.9800 |    922.6 tk
🔹 Hybrid            |    100.00% |   0.8800 |    891.9 tk
🔹 Hybrid+Rerank     |    100.00% |   1.0000 |    595.4 tk
---------------------------------------------------------------------------
💡 Tip: Reranker 사용 시 최종 전달 청크 수(top_n)에 따라 Avg Tokens가 최적화됩니다.

🧐 [리트리버별 Top-1 근거 비교 분석]
[1] 질문: Laycan 기간 중 Cancellation Clause를 행사하기 위한 선행 조건은?
🎯 기대 키워드: canceling date is not respected
----------------------------------------------------------------------------------------------------
🔍 [Base Vector  ] ✅ 일치
   > [Context: 용선 계약 가이드 - 계약 취소 및 취소일 규정 설명, 선박 미제공 시 계약 해지 조건]

14
Version 2.

{'Base Vector': {'HitRate': 1.0, 'MRR': 0.98, 'AvgTokens': 922.64},
 'Hybrid': {'HitRate': 1.0, 'MRR': 0.88, 'AvgTokens': 891.92},
 'Hybrid+Rerank': {'HitRate': 1.0, 'MRR': 1.0, 'AvgTokens': 595.44}}

# 📊 RAG 파이프라인 단계별 비교 분석 리포트

## 1. 종합 성능 비교표

| 지표 | **Base Vector** | **Hybrid (Vector+BM25)** | **Hybrid + Rerank** |
| --- | --- | --- | --- |
| **Hit Rate** | 100% | 100% | 100% |
| **MRR (순위 정확도)** | 0.9800 | 0.8800 | **1.0000** |
| **Avg Tokens** | 922.6 tk | 891.9 tk | **595.4 tk** |

---

## 2. 상세 비교 분석

### 🔹 1단계: Base Vector (의미 검색의 기초)

* **특징**: 질문의 핵심 의미(Semantic)를 파악해 관련 문서를 가져옵니다.
* **장점**: 의미 기반 최적화 - 단순히 텍스트만 있는 것이 아니라 "이 부분은 용어집의 A항목 섹션입니다"라는 정보가 본문에 박혀 있어, 벡터 검색만으로도 매우 높은 정확도(MRR 0.98)를 보입니다.
* **단점**: 관련된 5개의 청크를 필터링 없이 모두 LLM에게 전달하므로 **토큰 소모가 가장 큽니다.**
* **한계**: 질문 [6]번(AAAA)처럼 고유 명사의 형태가 중요한 경우, 의미(Semantic)만으로는 엉뚱한 '중재(Arbitration)' 청크를 가져오는 등 한계를 보입니다.

### 🔹 2단계: Hybrid (키워드 매칭의 노이즈와 역설)

* **특징**: 벡터 검색에 단어 매칭(BM25) 기술을 섞었습니다.
* **왜 MRR이 낮아졌을까?**:
* **키워드 간섭**: 질문에서 '풀네임 괄호 설명'이 빠지자, BM25가 `BBB`, `AAAA` 같은 짧은 약어를 문서 내의 다른 노이즈 단어들과 매칭시키며 정답 순위를 뒤로 밀어냈습니다.
* **실패 사례 분석**:
    * 약어의 함정 (BBB 질문): 질문의 BBB와 본문 내 선사명 BBC를 구분하지 못했습니다. 특히 모든 페이지 하단에 적힌 "© BBC Chartering" 문구 때문에 **84페이지(저작권 고지)가 정답보다 높은 순위로 올라오는 '키워드 노이즈'**가 대량 발생했습니다.
    * 도메인 특성: 해운 도메인 특유의 짧은 약어들이 일반 텍스트 내의 유사 철자와 겹치면서 검색 품질을 오히려 저하시켰습니다.
* **토큰**: BM25가 결과의 다양성을 확보해주면서 중복된 긴 문장들이 일부 제거되어 1단계에 비해 토큰은 소폭 감소했습니다.

### 🔹 3단계: Hybrid + Rerank (최종 최적화)

* **특징**: 앞서 찾은 후보군(5개)을 고성능 모델이 다시 읽고 순위를 매깁니다.
* **압도적 MRR (1.0)**: 하이브리드 단계에서 키워드 노이즈 때문에 2~3위로 밀려났던 정답들을 리랭커가 문맥을 정확히 파악해 **다시 1위로 고정**시켰습니다.
* **토큰 절감의 핵심**: 리랭커가 "이 청크는 질문과 상관없다"고 판단한 것들을 과감히 쳐내고 **'진짜 근거'만 LLM에게 전달**했습니다. 그 결과 토큰 사용량이 900대에서 600대 미만으로 **약 35% 이상 감소**하며 비용 효율성을 잡았습니다.

---

## 3. 핵심 인사이트

1. **Contextual Retrieval의 가치**: 청크 전처리에 LLM을 활용한 Anthropic 방식은 검색 엔진이 각 청크의 '정체성'을 파악하게 함으로써 Base Vector의 성능을 비약적으로 끌어올렸습니다.
2. **약어 매칭의 한계**: 해운 도메인처럼 약어가 많은 분야에서 단순 키워드 검색(BM25)은 오히려 독이 될 수 있음을 확인했습니다.
3. **Reranker의 필수성**: 검색 순위가 흔들려도 리랭커가 이를 최종적으로 보정해주기 때문에, 안정적인 답변 품질을 위해서는 리랭커 도입이 필수적입니다.
4. **경제성 달성**: 정확도 향상뿐만 아니라, **불필요한 컨텍스트 제거를 통한 비용 절감** 효과가 매우 뚜렷하게 나타났습니다.

---

## 추가 실험: RAG 생성 능력 실험 (테이블 생성)

In [69]:
def evaluate_generation_with_metrics(query, llm_model):
    # 1. 문서 검색 (우리의 최강 조합)
    retrieved_docs = hybrid_rerank_retriever.invoke(query)
    
    # [재료 확인용] LLM에게 전달될 원문 컨텍스트 생성
    context_list = [f"[문서 {i+1}]: {doc.page_content}" for i, doc in enumerate(retrieved_docs)]
    context = "\n\n".join(context_list)
    input_tokens = count_tokens(context)
    
    # ---------------------------------------------------------
    # 📦 [재료 확인 섹션] LLM에게 투입되는 원본 데이터 출력
    # ---------------------------------------------------------
    print("\n" + "🛒 [LLM에게 투입된 원재료 (Retrieved Context)]")
    print("-" * 80)
    for doc_content in context_list:
        print(doc_content)
        print("-" * 40) # 문서별 구분선
    print(f"📦 총 {len(retrieved_docs)}개의 문서가 검색되었습니다. (Total Tokens: {input_tokens})")
    print("=" * 80)
    
    # 2. 페르소나가 강화된 시스템 프롬프트
    system_prompt = """너는 세계 최고의 해운 물류 전문가야. 
    제공된 [Context]에 근거해서 사용자의 질문에 답해줘.
    
    [지침]
    1. 두 가지 이상의 개념을 비교할 때는 반드시 'Markdown Table' 형식을 사용해.
    2. 문서에 없는 내용은 절대로 지어내지 마 (Hallucination 방지).
    3. 답변 끝에는 참고한 문서 번호를 명시해줘.
    """
    
    user_message = f"[Context]\n{context}\n\n[Question]\n{query}"
    
    # 3. 답변 생성 (predict 대신 invoke 사용 및 .content 추출)
    # LangChain 최신 버전 규격에 맞게 수정되었습니다.
    raw_response = llm_model.invoke(f"{system_prompt}\n\n{user_message}")
    response = raw_response.content # 텍스트 내용만 추출
    
    output_tokens = count_tokens(response)
    
    # ---------------------------------------------------------
    # 📊 결과 리포트 출력
    # ---------------------------------------------------------
    print("\n" + "="*80)
    print(f"🚀 [RAG 생성 능력 테스트] 질문: {query}")
    print("="*80)
    print(response) # 실제 생성된 표 출력
    print("-"*80)
    
    print(f"📈 [성능 분석 데이터]")
    print(f"  • 검색 컨텍스트: {input_tokens} tk")
    print(f"  • LLM 생성 답변: {output_tokens} tk")
    print(f"  • 컨텍스트 압축률: {round(100 - (output_tokens/input_tokens*100), 1) if input_tokens > 0 else 0}%")
    print(f"  • 정보 밀도 지수: {round(output_tokens/input_tokens, 2) if input_tokens > 0 else 0}")
    print("="*80)

### 테스트 1: Liner Terms, Free In (FI), Free Out (FO) 관련 용어를 모두 모아서 잘 추론하는지 테스트

In [70]:
# 실험 1: 비용 부담 주체 비교
query_1 = "Liner Terms, Free In, Free Out 조건에서 선적 및 양하 비용 부담 차이를 표로 정리해줘."
evaluate_generation_with_metrics(query_1, answer_llm)


🛒 [LLM에게 투입된 원재료 (Retrieved Context)]
--------------------------------------------------------------------------------
[문서 1]: [Context: 운임조건 및 부가요금 규정, 선사별 요율 차이, 선적/하역 비용 구분 설명.]

72
Version 2.0 | July 2025
FIOS - Free in/out (loading/discharging is at consigner’s cost)
FIFO - Free in/Free out (vide FIOS)
FILO - Free in/Liner out (loading is at consigner’s cost, discharging is at liner cost)
LIFO - Liner in/Free out (loading is at liner cost, discharging is at consigner’s cost)
LILO - Liner in/out (loading and discharging is at liner cost)
Depending on the shipping line or a particular port practice the different surcharges can be 
added to rate:
CAF ( Currency Adjustment Factor ) - is a fee applied to the shipping costs to compensate 
for exchange rate fluctuations
BAF ( Bunker Adjustment Factor ) - refers to floating part of sea freight charges which 
represents additions due to oil prices
Wharfage is a port duty
----------------------------------------
[문서 2]: [Context: 용어집 섹션, 해

결과: 재료는 잘 가지고 왔지만 추론에서 Free In과 Free Out을 제대로 정리하지 못함

-> 추론 모델을 업그레이드 시켜보자

In [73]:
# 실험 1-1: 비용 부담 주체 비교, 모델 개선
upgraded_llm = ChatOpenAI(model="gpt-4.1", temperature=0)
query_1 = "Liner Terms, Free In, Free Out 조건에서 선적 및 양하 비용 부담 차이를 표로 정리해줘."
evaluate_generation_with_metrics(query_1, upgraded_llm)


🛒 [LLM에게 투입된 원재료 (Retrieved Context)]
--------------------------------------------------------------------------------
[문서 1]: [Context: 운임조건 및 부가요금 규정, 선사별 요율 차이, 선적/하역 비용 구분 설명.]

72
Version 2.0 | July 2025
FIOS - Free in/out (loading/discharging is at consigner’s cost)
FIFO - Free in/Free out (vide FIOS)
FILO - Free in/Liner out (loading is at consigner’s cost, discharging is at liner cost)
LIFO - Liner in/Free out (loading is at liner cost, discharging is at consigner’s cost)
LILO - Liner in/out (loading and discharging is at liner cost)
Depending on the shipping line or a particular port practice the different surcharges can be 
added to rate:
CAF ( Currency Adjustment Factor ) - is a fee applied to the shipping costs to compensate 
for exchange rate fluctuations
BAF ( Bunker Adjustment Factor ) - refers to floating part of sea freight charges which 
represents additions due to oil prices
Wharfage is a port duty
----------------------------------------
[문서 2]: [Context: 용어집 섹션, 해

모델을 바꾸니 추론 능력이 좋아져서 재료를 제대로 이해하고 정리함.

### 테스트 2: Demurrage, Detention, DeadFreight 관련 용어를 모두 모아서 잘 추론하는지 테스트

In [74]:
# 실험 2: 손해배상 개념 비교
query_2 = "Demurrage, Detention, Deadfreight의 발생 원인과 특징을 표로 정리해줘."
evaluate_generation_with_metrics(query_2, answer_llm)


🛒 [LLM에게 투입된 원재료 (Retrieved Context)]
--------------------------------------------------------------------------------
[문서 1]: [Context: 용어집 - 해운 법적 책임, 데드프레이트, 적재량, 선적 화물, 데드웨이트 화물, 선상 화물 관련 해운 용어 설명]

by a court. A court may either (1) set a detention rate as the same rate as demurrage if such 
a rate has been incorporated into the charter party or (2) base it on the daily running cost 
of the ship plus any profit which shipowner might reasonably have expected. 
Deadfreight - Damages payable by a shipper or charterer of a ship for failing to load the 
amount of cargo stipulated in the contract. Deadfreight is typically payable at the full freight 
rate.
Deadweight Cargo  - Cargo that is heavy in proportion to the space it uses. Freight for 
deadweight cargo is generally calculated on a per metric ton basis.  
Deck Cargo - Cargo carried on the open deck of a ship. A carrier and a shipper may agree to
----------------------------------------
[문서 2]: [Context: 용어집 해운 용어 설명: D/A(Disburs

전문가님의 판단이 정확합니다. 보고서의 생명은 **'정직함'**과 **'실패를 통한 개선 과정'**을 보여주는 것이니까요. 특히 미니 모델이 데이터를 **정반대(Inverted Logic)**로 해석했다는 점은 매우 중요한 실험적 발견입니다.

이 점을 명확히 강조하여 수정한 **'진실된' 보고서 요약**입니다.

---

## 📊 RAG 시스템 성능 평가 및 모델 비교 보고서

### 1. 모델별 생성 정확도 비교 (Crucial Discovery)

가장 중요한 **'해석 오류'** 사례를 중심으로 비교한 표입니다.

| 실험 항목 | 모델 체급 | 생성 결과 분석 | 판정 |
| --- | --- | --- | --- |
| **실험 1: FI/FO 비용 주체** | **GPT-4.1-mini** | **[치명적 오류]** 'Free'를 무료 서비스로 오해하여 **부담 주체를 정반대**로 답변 | **FAIL** |
| (동일 실험) | **GPT-4.1** | **[정상 해석]** 'Free of expense to owner'의 법적 논리를 파악하여 **정답** 답변 | **PASS** |
| **실험 2: 손해배상 개념** | **GPT-4.1-mini** | 용어 정의가 명확한 텍스트에서는 오해 없이 **정확히 요약** | **PASS** |

---

### 2. 기술적 원인 분석 (Root Cause)

* **미니 모델의 한계 (Pre-trained Bias)**: `Free`라는 단어의 지배적인 의미(공짜, 무료)가 문서에 적힌 전문적 맥락(면제)보다 우선순위로 작동함.
* **논리적 전치사 무시**: "Free of expense **to** shipowner"라는 구문에서 수혜 대상(`to shipowner`)을 정밀하게 매칭하지 못하고 키워드 위주로 해석함.

---

### 3. 향후 대응 전략 (Action Plan)

**모델 라우팅(Routing) 도입**:
* 비용, 법적 책임 등 **논리적 해석이 중요한 질문** → 고성능 모델(`gpt-4.1`) 강제 할당.
* 단순 용어 정의 및 일반 요약 → 효율적인 미니 모델 사용.



---

### 4. 종합 평가

> **"검색된 재료가 완벽하더라도, 소형 모델의 언어적 편향성이 오답을 유도할 수 있음을 확인. 실무 배포 시 특정 용어에 대한 사전 가이드 또는 상위 모델 혼용이 필수적임."**

---